### 1.  ERA5 data pulling

**Paso 1: Preparación del archivo de datos**

El primer paso consiste en generar un archivo en formato CSV que contenga los datos puntuales de balance de masa específico. Puedes usar excel, planillas de google o bloc de notas. Este archivo sera usado para crear el dataframe usado para el entrenamiento.

El archivo debe incluir las siguientes columnas:

- **POINT_ID**: Identificador único del punto de medición.
- **FROM_DATE**: Fecha de inicio del registro (formato: `YYYY-MM-DD`).
- **TO_DATE**: Fecha de fin del registro (formato: `YYYY-MM-DD`).
- **POINT_LON**: Latitud en grados decimales (WGS84).
- **POINT_LAT**: Longitud en grados decimales (WGS84).
- **POINT_ELEVATION**: Altitud en metros sobre el nivel del mar.
- **POINT_BALANCE**: Balance de masa específico (en mWE o unidad equivalente).

**Paso 2: Descargar datos del ERA5-land**
El siguiente paso es descargar los datos climatico de la region que se esta trabajando. Por como funciona el modelo, se consideraran las variables:
- 2m_temperature
- forecast_albedo
- surface_latent_heat_flux
- surface_net_thermal_radiation
- surface_solar_radiation_downwards
- total_precipitation

Para poder utilizar la API del Climate Data Store se tiene que seguir el paso a paso para instalar el [cliente](https://cds.climate.copernicus.eu/how-to-api). Bastan los primeros dos pasos.

**Step 1: Prepare the data file**

The first step is to create a CSV file containing point measurements of specific mass balance. You can use Excel, Google Sheets, or a plain-text editor. This file will be used to build the DataFrame for training.

The file must include the following columns:

- **POINT_ID**: Unique identifier of the measurement point.
- **FROM_DATE**: Start date of the record (format: `YYYY-MM-DD`).
- **TO_DATE**: End date of the record (format: `YYYY-MM-DD`).
- **POINT_LON**: Longitude in decimal degrees (WGS84).
- **POINT_LAT**: Latitude in decimal degrees (WGS84).
- **POINT_ELEVATION**: Elevation in meters above mean sea level.
- **POINT_BALANCE**: Specific mass balance (in m w.e. or equivalent).

**Step 2: Download ERA5-Land data**

Next, download the climate data for the region of interest. As required by the model, include the following variables:

- 2m_temperature
- forecast_albedo
- surface_latent_heat_flux
- surface_net_thermal_radiation
- surface_solar_radiation_downwards
- total_precipitation

To use the Climate Data Store API, follow the step-by-step instructions to install the [client](https://cds.climate.copernicus.eu/how-to-api). The first two steps are sufficient.


In [ ]:
import pandas as pd
import os
import warnings
from tqdm.notebook import tqdm
import zipfile
import cdsapi
import zipfile
import numpy as np
import glob
import xarray as xr

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

In [ ]:
#Ruta donde se almacenan los datos descargados
path_ERA5_raw: str = './era5land/raw/'
years: [str] = ["2000", "2001", "2002", "2003", "2004", "2005",]
months: [str] = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]

#Variables climaticas
variables [str] = [ "2m_temperature",
                "forecast_albedo",
                "surface_latent_heat_flux",
                "surface_net_thermal_radiation",
                "surface_sensible_heat_flux",
                "surface_solar_radiation_downwards",
                "total_precipitation"]

#0: lat max, 1: long min, 2: lat min, 3: long max  
area: [float] = [-45, -74, -55, -69]

In [ ]:
RUN = True
if RUN:
    os.makedirs(path_ERA5_raw, exist_ok=True)
    c = cdsapi.Client()
    c.retrieve(
        'reanalysis-era5-land-monthly-means', {
            'product_type': ['monthly_averaged_reanalysis'],
            "variable": variables,
            "year": years,
            "month": months,
            'time': ["00:00"],
            "data_format": "netcdf",
            "download_format": "zip",
            "area": area,
        }, path_ERA5_raw+'download.netcdf.zip')
    with zipfile.ZipFile(path_ERA5_raw+'download.netcdf.zip', 'r') as zip:
        zip.extractall(path_ERA5_raw)
    c.retrieve("reanalysis-era5-single-levels", {
            "product_type": ["reanalysis"],
            "variable": ["geopotential"],
            "year": ["2024"],
            "month": ["06"],
            "day": ["01"],
            "time": ["12:00"],
            "data_format": "netcdf"
        }, path_ERA5_raw+'era5_geopotential_pressure.nc')

In [ ]:
ds  = xr.open_dataset(path_ERA5_raw+'data_stream-moda.nc').drop_vars(("number", "expver"))
ds.rename({'valid_time': 'time'}).to_netcdf(path_ERA5_raw+"era5_monthly_averaged_data.nc")
ds

In [1]:
# %% 1) Imports & configuration
#  Imports and global configuration for the MassBalanceMachine workflow.
#  Importaciones y configuración global para el flujo de MassBalanceMachine.

import pandas as pd
import geopandas as gpd
import xarray as xr
import massbalancemachine as mbm

cfg = mbm.Config()

#  Central input path for stake-level target data (point mass balance).
#  Ruta central de entrada para los datos objetivo a nivel de estaca (balance de masa puntual).
target_data_fname = '/mnt/d/Investigadores/Duilio.Fonseca/Machine-Learning/data/input/point_mass_balance/master_input.csv'

In [4]:
# %% 2) Load raw stake data
#  Load raw CSV and preview a few rows to verify schema and basic content.
#  Cargar el CSV y previsualizar algunas filas para verificar el esquema y contenido básico.

data = pd.read_csv(target_data_fname)
print("Initial data preview:")
display(data.head(10))

Initial data preview:


,POINT_ID,FROM_DATE,TO_DATE,POINT_LON,POINT_LAT,POINT_ELEVATION,POINT_BALANCE
0,B1,2021-10-24,2022-06-04,-71.413294,-36.834797,2850.216,-11.34
1,B2,2022-01-15,2022-06-04,-71.412119,-36.834306,2884.160,-12.58
2,B3,2021-10-24,2022-06-04,-71.414980,-36.837341,2747.150,-8.67
3,B4,2021-10-24,2022-06-04,-71.414161,-36.836079,2795.900,-12.39
4,B5,2021-10-24,2022-06-04,-71.414799,-36.835262,2813.830,-3.43
5,B1,2022-10-22,2023-12-04,-71.412189,-36.834359,2877.330,-10.61
6,B2,2022-10-22,2023-12-04,-71.412701,-36.833693,2904.880,-11.77
7,B3,2022-10-22,2023-12-04,-71.414202,-36.835798,2796.720,-11.93
8,B4,2022-10-22,2023-12-04,-71.414925,-36.836251,2775.220,-11.35
9,B5,2022-10-22,2023-12-04,-71.412745,-36.836599,2787.540,-11.14


In [5]:
# %% 3) Parse and standardize dates
#  Parse ISO-like dates, derive YEAR, and store dates as yyyymmdd strings for downstream tools.
#  Parsear fechas tipo ISO, derivar YEAR y guardar fechas como cadenas yyyymmdd para herramientas posteriores.

# Assuming format is YYYY-MM-DD (standard ISO), so no dayfirst=True
data['TO_DATE_parsed'] = pd.to_datetime(data["TO_DATE"], errors="coerce")
data['FROM_DATE_parsed'] = pd.to_datetime(data["FROM_DATE"], errors="coerce")

# Extract year and reformat as yyyymmdd
data['YEAR'] = data['TO_DATE_parsed'].dt.year
data["TO_DATE"] = data['TO_DATE_parsed'].dt.strftime("%Y%m%d")
data["FROM_DATE"] = data['FROM_DATE_parsed'].dt.strftime("%Y%m%d")

# Drop temporary parsed columns if not needed
data = data.drop(columns=["TO_DATE_parsed", "FROM_DATE_parsed"])

print("After processing:")
display(data.head(10))

#  Quick NA diagnostics for date fields; filter to keep only valid rows if desired.
#  Diagnóstico rápido de NAs en fechas; filtrar para conservar solo filas válidas si se desea.

print(f"Rows with missing FROM_DATE: {data['FROM_DATE'].isnull().sum()}")
print(f"Rows with missing TO_DATE: {data['TO_DATE'].isnull().sum()}")
print(f"Rows with missing YEAR: {data['YEAR'].isnull().sum()}")
data = data.dropna(subset=["FROM_DATE", "TO_DATE", "YEAR"])



After processing:


,POINT_ID,FROM_DATE,TO_DATE,POINT_LON,POINT_LAT,POINT_ELEVATION,POINT_BALANCE,YEAR
0,B1,20211024,20220604,-71.413294,-36.834797,2850.216,-11.34,2022
1,B2,20220115,20220604,-71.412119,-36.834306,2884.160,-12.58,2022
2,B3,20211024,20220604,-71.414980,-36.837341,2747.150,-8.67,2022
3,B4,20211024,20220604,-71.414161,-36.836079,2795.900,-12.39,2022
4,B5,20211024,20220604,-71.414799,-36.835262,2813.830,-3.43,2022
5,B1,20221022,20231204,-71.412189,-36.834359,2877.330,-10.61,2023
6,B2,20221022,20231204,-71.412701,-36.833693,2904.880,-11.77,2023
7,B3,20221022,20231204,-71.414202,-36.835798,2796.720,-11.93,2023
8,B4,20221022,20231204,-71.414925,-36.836251,2775.220,-11.35,2023
9,B5,20221022,20231204,-71.412745,-36.836599,2787.540,-11.14,2023


Rows with missing FROM_DATE: 0
Rows with missing TO_DATE: 0
Rows with missing YEAR: 0


In [6]:

# %% 4) Load glacier outlines and map RGI IDs
#  Load glacier polygons (RGI-derived) and assign RGIId to each stake measurement.
#  Cargar polígonos de glaciares (derivados de RGI) y asignar RGIId a cada medición de estaca.

glacier_outline_fname = "/mnt/d/Investigadores/Duilio.Fonseca/Machine-Learning/data/input/shapes/RGI_chile_v2/RGI_chile_v2.shp"
glacier_outline = gpd.read_file(glacier_outline_fname)
display(glacier_outline.head())

#  Spatial join/mapping from stake coordinates to glacier RGI footprint.
#  Unión/mapeo espacial desde coordenadas de estaca al polígono RGI del glaciar.
data = mbm.data_processing.utils.get_rgi(data=data, glacier_outlines=glacier_outline)
display(data)

,RGIId,GLIMSId,BgnDate,EndDate,CenLon,CenLat,O1Region,O2Region,Area,Zmin,...,Aspect,Lmax,Status,Connect,Form,TermType,Surging,Linkages,Name,geometry
0,RGI60-17.01218,G289664E34607S,20009999,20030531,-70.3361,-34.6072,17,2,110.978,2441,...,121,11211,0,0,0,0,3,9,Universidad,"POLYGON ((-70.3996 -34.61568, -70.39993 -34.61..."
1,RGI60-17.12440,G287979E39946S,20009999,20030531,-72.0213,-39.9458,17,2,6.316,1567,...,149,3499,0,0,1,0,9,9,Mocho S,"POLYGON ((-72.00482 -39.94868, -72.00541 -39.9..."
2,RGI60-17.12442,G287988E39929S,20009999,20030531,-72.0124,-39.9293,17,2,4.767,1672,...,78,2235,0,0,0,0,9,9,Mocho N,"POLYGON ((-72.01443 -39.91445, -72.01477 -39.9..."
3,RGI60-17.13045,G288585E36836S,20009999,20030531,-71.4149,-36.8356,17,2,1.256,2472,...,212,1884,0,0,1,0,9,9,None,"POLYGON ((-71.40594 -36.83148, -71.40592 -36.8..."
4,RGI60-17.14088,G289902E33030S,20009999,20030531,-70.0984,-33.0296,17,2,8.376,2929,...,358,8560,0,0,0,0,3,9,Gl Juncal Norte,"POLYGON ((-70.10482 -33.05111, -70.10508 -33.0..."


,POINT_ID,FROM_DATE,TO_DATE,POINT_LON,POINT_LAT,POINT_ELEVATION,POINT_BALANCE,YEAR,RGIId
0,B1,20211024,20220604,-71.413294,-36.834797,2850.216,-11.340,2022,RGI60-17.13045
1,B2,20220115,20220604,-71.412119,-36.834306,2884.160,-12.580,2022,RGI60-17.13045
2,B3,20211024,20220604,-71.414980,-36.837341,2747.150,-8.670,2022,RGI60-17.13045
3,B4,20211024,20220604,-71.414161,-36.836079,2795.900,-12.390,2022,RGI60-17.13045
4,B5,20211024,20220604,-71.414799,-36.835262,2813.830,-3.430,2022,RGI60-17.13045
...,...,...,...,...,...,...,...,...,...
813,12-awsfurg,20181120,20190423,-70.867957,-54.398644,207.000,-4.590,2019,RGI60-17.03160
814,12-awsfurg,20190423,20200313,-70.872567,-54.397850,188.000,-6.534,2020,RGI60-17.03160
815,1-northbal,20190423,20200313,-70.869750,-54.396694,197.000,-6.300,2020,RGI60-17.03160
816,1-southbal,20190423,20200313,-70.869505,-54.398830,202.000,-6.120,2020,RGI60-17.03160


In [7]:
# %% 5) Instantiate MBM Dataset and add topography
#  Create the MBM Dataset for Chile (region_name='CL') and attach local file base path.
#  Crear el Dataset de MBM para Chile (region_name='CL') y adjuntar la ruta base local de archivos.

dataset = mbm.Dataset(cfg=cfg, data=data, region_name='CL',
                      data_path='/mnt/d/Investigadores/Duilio.Fonseca/Machine-Learning/data/files')

#  Define topographic variables to fetch (see OGGM docs for availability).
#  Definir variables topográficas a recuperar (ver documentación de OGGM para disponibilidad).
voi_topographical = ['aspect', 'slope']

#  Sample topography at stake coordinates and append as features to the dataset.
#  Muestrear topografía en las coordenadas de las estacas y añadir como características al dataset.
dataset.get_topo_features(vois=voi_topographical)

2025-08-27 11:59:51: oggm.cfg: Reading default parameters from the OGGM `params.cfg` configuration file.
2025-08-27 11:59:51: oggm.cfg: Multiprocessing switched OFF according to the parameter file.
2025-08-27 11:59:51: oggm.cfg: Multiprocessing: using all available processors (N=40)
2025-08-27 11:59:54: oggm.cfg: PARAMS['border'] changed from `80` to `10`.
2025-08-27 11:59:54: oggm.cfg: Multiprocessing switched ON after user settings.
2025-08-27 11:59:54: oggm.cfg: PARAMS['continue_on_error'] changed from `False` to `True`.
2025-08-27 11:59:55: oggm.workflow: init_glacier_directories from prepro level 3 on 12 glaciers.
2025-08-27 11:59:55: oggm.workflow: Execute entity tasks [gdir_from_prepro] on 12 glaciers
2025-08-27 11:59:57: oggm.workflow: Execute entity tasks [gridded_attributes] on 12 glaciers



[DEBUG] Starting retrieval of topographical features...
[DEBUG] Variables of interest: ['aspect', 'slope']
[DEBUG] Number of glacier directories: 12

[DEBUG] Processing glacier 1/12 - RGI ID: RGI60-17.13045
[DEBUG] Number of stake points: 17
[DEBUG] Retrieved topographical data shape (1:1): (17, 4)
[DEBUG] Sample of retrieved data:
     aspect     slope         x           y
0  4.722618  0.446868 -62.42189 -4075701.75
1  4.722618  0.446868 -62.42189 -4075701.75
2  4.722618  0.446868 -62.42189 -4075701.75
3  4.722618  0.446868 -62.42189 -4075701.75
4  4.722618  0.446868 -62.42189 -4075701.75
[DEBUG] Updated DataFrame rows for glacier RGI60-17.13045: 17

[DEBUG] Processing glacier 2/12 - RGI ID: RGI60-17.14887
[DEBUG] Number of stake points: 14
[DEBUG] Retrieved topographical data shape (1:1): (14, 4)
[DEBUG] Sample of retrieved data:
    aspect    slope          x          y
0  4.50176  0.25894 -69.276581 -3240566.5
1  4.50176  0.25894 -69.276581 -3240566.5
2  4.50176  0.25894 -69.2765

In [ ]:
# %% 6) Fix ERA5 NetCDF variable names (one-off preprocessing)
#  Open ERA5 dataset, rename variables as needed for MBM compatibility, and save a fixed copy.
#  Abrir el dataset ERA5, renombrar variables según sea necesario para compatibilidad MBM y guardar una copia corregida.

era5_path = '/mnt/d/Investigadores/Duilio.Fonseca/Machine-Learning/data/input/ERA5/CL_era5_land.nc'
ds = xr.open_dataset(era5_path)
print("Original variabl", list(ds.variables))

# Rename 'valid_time' to 'time' and drop unused dimensions if present
ds = ds.rename({'valid_time': 'time'})
ds = ds.drop_vars(("expver", "number"))  # safe if they exist; ignored otherwise

out_path = '/mnt/d/Investigadores/Duilio.Fonseca/Machine-Learning/data/input/ERA5/CL_era5_land_fixed.nc'
ds.to_netcdf(out_path)
print(f"✓ Variable renamed and saved to: {out_path}")

In [ ]:

# %% 7) Attach climate datasets and extract features
#  Point each stake to ERA5Land and geopotential fields; compute climate features per measurement.
#  Vincular cada estaca a ERA5Land y geopotencial; calcular características climáticas por medición.

era5_climate_data = out_path
geopotential_data = '/mnt/d/Investigadores/Duilio.Fonseca/Machine-Learning/data/input/ERA5/geopotential/era5_geopotential_pressure.nc'

dataset.get_climate_features(climate_data=era5_climate_data, geopotential_data=geopotential_data)

#  Basic NA handling—inspect, then drop incomplete rows for a clean modeling table.
#  Manejo básico de NAs—inspeccionar y luego eliminar filas incompletas para una tabla limpia de modelado.
dataset.data.isnull().sum().sort_values(ascending=False)
dataset.data = dataset.data.dropna().reset_index(drop=True)
dataset.data.isnull().sum().sort_values(ascending=False)

In [ ]:

# %% 8) Monthly aggregation and final preview
#  Define climate variable short names and convert records to monthly resolution.
#  Definir nombres cortos de variables climáticas y convertir los registros a resolución mensual.

vois_climate = ['t2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str']

#  Aggregate stake-level signals and features to monthly scale for modeling.
#  Agregar señales y variables a escala mensual para el modelado.
dataset.convert_to_monthly(vois_climate=vois_climate, vois_topographical=voi_topographical)

#  Inspect the final modeling table (one row per stake-month with features/targets).
#  Inspeccionar la tabla final de modelado (una fila por estaca-mes con características/objetivos).
display(dataset.data)
